To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)



<b>📢 لا تنسَ الاشتراك في قناة مدرسة الذكاء الاصطناعي ومتابعتها على جميع منصات التواصل الاجتماعي للحصول على أحدث الدروس والشروحات في الذكاء الاصطناعي والتعلم الآلي.</b>

<div class="align-center">
<a href="https://www.youtube.com/@ArabianAiSchool"><img src="https://img.shields.io/badge/YouTube-Subscribe-red?style=for-the-badge&logo=youtube" width="145"></a>
<a href="https://www.facebook.com/arabianaischool/"><img src="https://img.shields.io/badge/Facebook-Follow-blue?style=for-the-badge&logo=facebook" width="145"></a>
<a href="http://instagram.com/arabianaischool"><img src="https://img.shields.io/badge/Instagram-Follow-purple?style=for-the-badge&logo=instagram" width="145"></a>
<a href="https://x.com/arabianaischool"><img src="https://img.shields.io/badge/X-Follow-black?style=for-the-badge&logo=x" width="145"></a>


# **تثبيت مكتبة ان سلوث والمكتبات الأساسية**

**في الخلية دي بنجهز بيئة التشغيل اللي هنشتغل عليها.**

**أول حاجة بنثبت مكتبة ان سلوث، ودي مكتبة بتساعدنا ندرّب موديلات الذكاء الاصطناعي بكفاءة أعلى، خصوصاً لما نكون شغالين على كرت شاشة محدود زي اللي بيكون موجود في جوجل كولاب.**

**بعد كده بنثبت شوية مكتبات مهمة للتدريب، وتحميل الداتا، والتعامل مع هاجينج فيس، وتشغيل موديل سبارك تي تي اس.**

**الخلية دي تعتبر مرحلة تجهيز الأدوات قبل ما نبدأ نحمّل الموديل أو نجهز الداتا.**

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!git clone https://github.com/SparkAudio/Spark-TTS
!pip install omegaconf einx torchcodec "datasets>=3.4.1,<4.0.0"

# **تحميل نموذج سبارك تي تي اس**

**في الخلية دي بنحمّل كود مشروع سبارك تي تي اس من جيت هاب.**

**سبارك تي تي اس مش مجرد موديل واحد وخلاص، لكنه مشروع كامل فيه كود لتحويل الصوت لتوكنز، وكود للتعامل مع الموديل، وكود يساعدنا نرجع التوكنز لصوت مسموع مرة تانية.**

**كمان بنثبت مكتبات مهمة لمعالجة الصوت والداتا، لأن تدريب موديل صوت محتاج أدوات مختلفة عن تدريب موديل نص فقط.**

In [ ]:
!git clone https://github.com/SparkAudio/Spark-TTS
!pip install omegaconf einx torchcodec "datasets>=3.4.1,<4.0.0"

fatal: destination path 'Spark-TTS' already exists and is not an empty directory.


### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

Thank you to [Etherl](https://huggingface.co/Etherll) for creating this notebook!

# **تثبيت مكتبة تورش فيجن**

**في الخلية دي بنثبت مكتبة تورش فيجن بنسخة مناسبة لبيئة التشغيل.**

**ممكن الاسم يحسسك إنها مكتبة خاصة بالصور فقط، لكن بعض مكتبات بايتورش والمكونات المساعدة بتحتاجها عشان تشتغل بدون أخطاء.**

**الهدف من الخلية دي مش تحسين جودة الصوت مباشرة، لكنها بتساعدنا نتجنب مشاكل توافق أثناء تشغيل النوتبوك.**

In [ ]:
!uv pip install torchvision>=0.24.0 --system

Using Python 3.12.13 environment at: /usr
Checked 1 package in 187ms


# **تحميل موديل سبارك تي تي اس الأساسي**



**هنا بنحمّل موديل سبارك تي تي اس الجاهز من هاجينج فيس، وتحديداً جزء اللغة اللي هنعدله أثناء التدريب.**

**إحنا مش بنبني موديل من الصفر، لأن بناء موديل صوت من الصفر محتاج داتا ضخمة وتكلفة عالية جداً.**

**اللي بنعمله هنا إننا بنبدأ من موديل جاهز أصلاً، وبعد كده نعمله فاين تيونينج على الداتا اللي عندنا.**

**في التجربة دي إحنا عاملين تدريب كامل، يعني بنسمح بتعديل أوزان الموديل نفسه. ده ممكن يدي تأثير أقوى، لكنه يحتاج ذاكرة أكبر ووقت أكثر.**

In [ ]:
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

max_seq_length = 2048 # Choose any for long context!

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Qwen3 new models
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    # Other very popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

# Download model and code
snapshot_download("SparkAudio/Spark-TTS-0.5B", local_dir = "Spark-TTS-0.5B")

model, tokenizer = FastModel.from_pretrained(
    model_name = "Spark-TTS-0.5B/LLM",
    max_seq_length = max_seq_length,
    dtype = torch.float32, # Spark seems to only work on float32 for now
    full_finetuning = True, # We support full finetuning now!
    load_in_4bit = False,
    #token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


.gitattributes: 0.00B [00:00, ?B/s]

config.yaml: 0.00B [00:00, ?B/s]

BiCodec/model.safetensors:   0%|          | 0.00/626M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

wav2vec2-large-xlsr-53/pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


# **تسجيل الدخول إلى هاجينج فيس**

**في الخلية دي بنسجل الدخول إلى هاجينج فيس باستخدام توكن محفوظ داخل كولاب.**

**هاجينج فيس هو المكان اللي بنحمّل منه الموديلات والداتا، وكمان ممكن نرفع عليه الداتا المتجهزة أو الموديل النهائي بعد التدريب.**

**مهم جداً إن التوكن ما يتكتبش مباشرة داخل النوتبوك، خصوصاً لو هتصور الشاشة أو تشارك الملف مع الناس.**

**عشان كده بنحفظ التوكن في كولاب، وبنخلي الكود يقرأه من هناك.**


In [ ]:
#@title Hugging Face Login from Colab Secrets

import os
from google.colab import userdata
from huggingface_hub import login, whoami

# لازم يكون اسم السر في Colab Secrets هو HF_TOKEN بالظبط
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "لم أجد HF_TOKEN في Colab Secrets. "
        "تأكد إن اسم السر HF_TOKEN وإنك فعلت access للـ notebook."
    )

# تنظيف التوكن من أي مسافات أو علامات اقتباس
HF_TOKEN = HF_TOKEN.strip().replace('"', '').replace("'", "")

# لو بالغلط حطيته بصيغة Bearer hf_xxx
if HF_TOKEN.lower().startswith("bearer "):
    HF_TOKEN = HF_TOKEN.split(" ", 1)[1].strip()

print("Token starts with:", HF_TOKEN[:6])
print("Token length:", len(HF_TOKEN))

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("التوكن لا يبدأ بـ hf_، غالبا Secret غلط أو فاضي.")

# نخزنه في البيئة عشان trainer / push_to_hub يستخدمه
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# تسجيل الدخول
login(token=HF_TOKEN, add_to_git_credential=True)

# اختبار فعلي إن التوكن صالح
info = whoami(token=HF_TOKEN)
print("Logged in as:", info["name"])

Token starts with: hf_KpK
Token length: 37
Logged in as: testbb


# **تحميل الداتا العربية**

**في الخلية دي بنحمّل داتا عربية من هاجينج فيس.**

**داتا تدريب الصوت بتكون غالباً عبارة عن أمثلة كثيرة، وكل مثال فيه نص مكتوب وصوت بيقرأ نفس النص.**

**الموديل بيتعلم من العلاقة بين النص والصوت.**

**يعني يتعلم الكلمة دي اتنطقت إزاي، والجملة دي اتقالت بسرعة ولا بهدوء، وفين الوقفات، وطريقة النطق شكلها إيه.**

**هنا كمان بنضيف عمود للنص المنظف اللي هنستخدمه في التدريب.**

**في الداتا دي التنظيف بسيط، لأن الداتا أصلها عربية ومجهزة نسبياً. لكن لو استخدمنا داتا من تسجيلاتنا أو فيديوهاتنا، غالباً هنحتاج تنظيف أقوى، زي تحويل الأرقام لكلام، وتعريب الكلمات الإنجليزية، وحذف الرموز الغريبة.**


In [ ]:
#@title Load ClArTTS Arabic Dataset

from datasets import load_dataset
import re

RAW_DATASET_ID = "MBZUAI/ClArTTS" # هنا ضع الداتا سيت التي تريدها
TOKENIZED_DATASET_ID = "testbb/spark_tts_clartts_arabic_tokens_v1" # هنا استبدل الاسم الخاص بك
FINETUNED_MODEL_ID = "testbb/spark-tts-clartts-arabic-v1" # هنا استبدل الاسم الخاص بك

# Use only train split for training.
# ClArTTS train has 9500 rows; test has 205 rows.
dataset = load_dataset(RAW_DATASET_ID, split="train")

print(dataset)
print("Columns:", dataset.column_names)
print("First example keys:", dataset[0].keys())
print("First text:")
print(dataset[0]["text"])
print("Sampling rate:", dataset[0]["sampling_rate"])
print("Duration:", dataset[0]["duration"])

def normalize_text_for_clartts(text):
    # Keep Arabic diacritics. Only clean spaces and invisible characters.
    text = "" if text is None else str(text)
    text = text.replace("ـ", "")
    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

dataset = dataset.map(
    lambda x: {"normalized_text": normalize_text_for_clartts(x["text"])},
    desc="Adding normalized_text"
)

print("\nNormalized text preview:")
print(dataset[0]["normalized_text"])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

data/train-00000-of-00026.parquet:   0%|          | 0.00/123M [00:00<?, ?B/s]

data/train-00001-of-00026.parquet:   0%|          | 0.00/121M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/205 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

Dataset({
    features: ['text', 'file', 'audio', 'sampling_rate', 'duration'],
    num_rows: 9500
})
Columns: ['text', 'file', 'audio', 'sampling_rate', 'duration']
First example keys: dict_keys(['text', 'file', 'audio', 'sampling_rate', 'duration'])
First text:
لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّ مِنْ بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌ
Sampling rate: 40100
Duration: 5.29


Adding normalized_text:   0%|          | 0/9500 [00:00<?, ? examples/s]


Normalized text preview:
لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّ مِنْ بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌ


# **تحويل الصوت إلى توكنز صوتية**

**دي من أهم الخلايا في تجهيز الداتا.**

**الموديل لا يسمع الصوت زي الإنسان. إحنا بنسمع الصوت ككلام، لكن الكمبيوتر والموديل يتعاملوا مع أرقام وتمثيلات رقمية.**

**عشان كده في الخلية دي بنحوّل كل ملف صوتي إلى توكنز صوتية يفهمها سبارك تي تي اس.**

**بعد كده بنربط التوكنز الصوتية بالنص المقابل لها، ونبني عينة تدريب بالشكل اللي الموديل يقدر يتعلم منه.**

**الفكرة ببساطة إننا لا نرمي ملف الصوت الخام للموديل ونقول له اتعلم. لازم الأول نحوله لشكل منظم مناسب للتدريب.**

**في آخر الخلية بننقل بعض المكونات إلى المعالج العادي ونفضي ذاكرة كرت الشاشة، لأن مرحلة التوكنة ممكن تستهلك ذاكرة كبيرة، وبعدها نحتاج مساحة لمرحلة التدريب.**

In [ ]:
#@title Tokenization Function For ClArTTS

import os
import re
import sys
import torch
import numpy as np
import torchaudio.transforms as T

if "Spark-TTS" not in sys.path:
    sys.path.append("Spark-TTS")

from sparktts.models.audio_tokenizer import BiCodecTokenizer
from sparktts.utils.audio import audio_volume_normalize

LOCAL_SNAPSHOT_DIR = "Spark-TTS-0.5B"

audio_tokenizer = BiCodecTokenizer(LOCAL_SNAPSHOT_DIR, "cuda")

def extract_wav2vec2_features(wavs: torch.Tensor) -> torch.Tensor:
    """Extract Wav2Vec2 hidden-state features required by Spark-TTS BiCodec."""
    if wavs.shape[0] != 1:
        raise ValueError(f"Expected batch size 1, but got shape {wavs.shape}")

    wav_np = wavs.squeeze(0).detach().cpu().numpy()

    processed = audio_tokenizer.processor(
        wav_np,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )

    input_values = processed.input_values.to(audio_tokenizer.feature_extractor.device)

    with torch.no_grad():
        model_output = audio_tokenizer.feature_extractor(
            input_values,
            output_hidden_states=True,
        )

    if model_output.hidden_states is None:
        raise ValueError("Wav2Vec2Model did not return hidden states.")

    num_layers = len(model_output.hidden_states)
    required_layers = [11, 14, 16]
    if any(layer >= num_layers for layer in required_layers):
        raise IndexError(
            f"Requested hidden states {required_layers}, but model has only {num_layers} layers."
        )

    feats_mix = (
        model_output.hidden_states[11]
        + model_output.hidden_states[14]
        + model_output.hidden_states[16]
    ) / 3

    return feats_mix

def get_audio_array_and_sr(example):
    """
    ClArTTS stores audio as a list and has a separate sampling_rate column.
    Other HF audio datasets may store audio as {'array': ..., 'sampling_rate': ...}.
    This function supports both formats.
    """
    audio_obj = example["audio"]

    if isinstance(audio_obj, dict):
        audio_array = audio_obj["array"]
        sampling_rate = audio_obj["sampling_rate"]
    else:
        audio_array = audio_obj
        sampling_rate = example.get("sampling_rate", 40100)

    audio_array = np.asarray(audio_array, dtype=np.float32)

    if audio_array.ndim > 1:
        audio_array = np.mean(audio_array, axis=0).astype(np.float32)

    return audio_array, int(sampling_rate)

def formatting_audio_func(example):
    # Use normalized_text added in the previous cell. Keep diacritics.
    text = example.get("normalized_text") or example.get("text") or ""
    text = str(text).strip()

    audio_array, sampling_rate = get_audio_array_and_sr(example)

    target_sr = audio_tokenizer.config["sample_rate"]

    # Resample full audio to Spark-TTS expected sample rate.
    if sampling_rate != target_sr:
        resampler = T.Resample(orig_freq=sampling_rate, new_freq=target_sr)
        audio_tensor_temp = torch.from_numpy(audio_array).float()
        audio_array = resampler(audio_tensor_temp).numpy().astype(np.float32)

    if audio_tokenizer.config.get("volume_normalize", False):
        audio_array = audio_volume_normalize(audio_array).astype(np.float32)

    # IMPORTANT:
    # wav must be the FULL audio.
    # ref_wav is only a short reference clip.
    # Do not train on ref_wav as if it were the full audio.
    audio_tensor = torch.from_numpy(audio_array).unsqueeze(0).float().to(audio_tokenizer.device)

    ref_wav_np = audio_tokenizer.get_ref_clip(audio_array)
    ref_wav_tensor = torch.from_numpy(ref_wav_np).unsqueeze(0).float().to(audio_tokenizer.device)

    # Wav2Vec2 features must be extracted from the full audio at 16 kHz.
    if target_sr != 16000:
        resampler_16k = T.Resample(orig_freq=target_sr, new_freq=16000).to(audio_tokenizer.device)
        wav_for_feat = resampler_16k(audio_tensor)
    else:
        wav_for_feat = audio_tensor

    feat = extract_wav2vec2_features(wav_for_feat)

    batch = {
        "wav": audio_tensor,
        "ref_wav": ref_wav_tensor,
        "feat": feat.to(audio_tokenizer.device),
    }

    with torch.no_grad():
        semantic_token_ids, global_token_ids = audio_tokenizer.model.tokenize(batch)

    global_ids = global_token_ids.squeeze().detach().cpu().numpy().tolist()
    semantic_ids = semantic_token_ids.squeeze().detach().cpu().numpy().tolist()

    global_tokens = "".join([f"<|bicodec_global_{int(i)}|>" for i in global_ids])
    semantic_tokens = "".join([f"<|bicodec_semantic_{int(i)}|>" for i in semantic_ids])

    training_text = "".join([
        "<|task_tts|>",
        "<|start_content|>",
        text,
        "<|end_content|>",
        "<|start_global_token|>",
        global_tokens,
        "<|end_global_token|>",
        "<|start_semantic_token|>",
        semantic_tokens,
        "<|end_semantic_token|>",
        "<|im_end|>",
    ])

    return {"text": training_text}

# Remove all raw columns after converting to the single training text field.
columns_to_remove = list(dataset.column_names)

dataset = dataset.map(
    formatting_audio_func,
    remove_columns=columns_to_remove,
    desc="Converting ClArTTS audio to Spark tokens",
)

print("Tokenization finished.")
print(dataset)
print("Training text preview:")
print(dataset[0]["text"][:500])

print("Moving BiCodec model and Wav2Vec2Model to CPU.")
audio_tokenizer.model.cpu()
audio_tokenizer.feature_extractor.cpu()
torch.cuda.empty_cache()

Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


Parameter 'function'=<function formatting_audio_func at 0x7d0b327eb100> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Converting ClArTTS audio to Spark tokens:   0%|          | 0/9500 [00:00<?, ? examples/s]

Tokenization finished.
Dataset({
    features: ['text'],
    num_rows: 9500
})
Training text preview:
<|task_tts|><|start_content|>لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّ مِنْ بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌ<|end_content|><|start_global_token|><|bicodec_global_2563|><|bicodec_global_2099|><|bicodec_global_880|><|bicodec_global_3375|><|bicodec_global_2400|><|bicodec_global_2380|><|bicodec_global_1044|><|bicodec_global_1607|><|bicodec_global_3535|><|bicodec_global_3365|><|bicodec_global_3635|><|bicodec_global_3248|><|bicodec_global_815|><|bicodec_global_3605|><|bicodec_global_3315|><|bi
Moving BiCodec model and Wav2Vec2Model to CPU.


# **رفع الداتا المتجهزة إلى هاجينج فيس**

**بعد ما خلصنا تحويل الصوت إلى توكنز، بنرفع الداتا المتجهزة على هاجينج فيس.**

**السبب إن خطوة التوكنة ممكن تكون طويلة وثقيلة، ولو جلسة كولاب فصلت هنضطر نعيدها من البداية.**

**لما نرفع الداتا المتجهزة، نقدر في أي جلسة جديدة نحملها مباشرة ونبدأ التدريب من غير ما نعيد تجهيز الصوت.**

**الخطوة دي توفر وقت كبير جداً لو هتكرر التجربة أو تغير إعدادات التدريب أكثر من مرة.**

**الرفع يحتاج حساب هاجينج فيس وتوكن عنده صلاحية كتابة.**

In [ ]:
# Push tokenized ClArTTS dataset so you do not need to tokenize again.
TOKENIZED_DATASET_ID = "testbb/spark_tts_clartts_arabic_tokens_v1" # غير اسم اليوزر الى حسابك

dataset.push_to_hub(TOKENIZED_DATASET_ID, private=True)

print("Pushed tokenized dataset to:", TOKENIZED_DATASET_ID)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 13.1MB / 13.1MB            

README.md:   0%|          | 0.00/277 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed tokenized dataset to: testbb/spark_tts_clartts_arabic_tokens_v1


# **تحميل الداتا المتجهزة**

**الخلية دي نستخدمها لو الداتا المتوكّنة اترفعت قبل كده على هاجينج فيس.**

**بدل ما نعيد تحويل الصوت لتوكنز من جديد، بنحمّل النسخة الجاهزة مباشرة.**

**دي خطوة مفيدة جداً لو جلسة كولاب فصلت، أو لو عايز تعيد التدريب بإعدادات مختلفة من غير ما تعيد تجهيز الداتا.**

**يعني عندنا طريقين. إما نجهز الداتا من البداية، أو لو جهزناها قبل كده نحمّل النسخة الجاهزة مباشرة.**

In [ ]:
from datasets import load_dataset

TOKENIZED_DATASET_ID = "testbb/spark_tts_clartts_arabic_tokens_v1" # غير اسم اليوزر الى حسابك

dataset = load_dataset(TOKENIZED_DATASET_ID, split="train")

print(dataset)
print(dataset[0]["text"][:500])

data/train-00000-of-00001.parquet:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9500 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 9500
})
<|task_tts|><|start_content|>لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّ مِنْ بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌ<|end_content|><|start_global_token|><|bicodec_global_2563|><|bicodec_global_2099|><|bicodec_global_880|><|bicodec_global_3375|><|bicodec_global_2400|><|bicodec_global_2380|><|bicodec_global_1044|><|bicodec_global_1607|><|bicodec_global_3535|><|bicodec_global_3365|><|bicodec_global_3635|><|bicodec_global_3248|><|bicodec_global_815|><|bicodec_global_3605|><|bicodec_global_3315|><|bi


# **اختبار جودة التوكنة قبل التدريب**

**قبل ما نبدأ التدريب، لازم نتأكد إن تحويل الصوت إلى توكنز تم بشكل صحيح.**

**في الخلية دي بناخد عينات من الداتا المتجهزة، ونحاول نرجع التوكنز الصوتية إلى صوت مسموع.**

**لو الصوت الناتج مقبول، ده معناه إن خطوة التوكنة غالباً سليمة.**

**لو الصوت هنا فيه مشكلة كبيرة، يبقى التدريب بعد كده هيكون مبني على داتا غلط.**

**دي خطوة أمان مهمة جداً، لأن التدريب كله بيعتمد على جودة الداتا المتجهزة.**


In [ ]:
#@title Check Spark Tokenization Quality Before Training

import os
import re
import sys
import random
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio as IPythonAudio, display

# تحديد مسار موديل Spark-TTS
if "LOCAL_SNAPSHOT_DIR" not in globals():
    if os.path.exists("Spark-TTS-0.5B"):
        LOCAL_SNAPSHOT_DIR = "Spark-TTS-0.5B"
    else:
        LOCAL_SNAPSHOT_DIR = snapshot_download(
            "SparkAudio/Spark-TTS-0.5B",
            local_dir="Spark-TTS-0.5B"
        )

print("LOCAL_SNAPSHOT_DIR =", LOCAL_SNAPSHOT_DIR)

# تأكد إن كود Spark-TTS موجود في path
if "Spark-TTS" not in sys.path:
    sys.path.append("Spark-TTS")

# تحميل audio_tokenizer لو مش موجود
if "audio_tokenizer" not in globals():
    print("audio_tokenizer غير موجود. سيتم تحميله من Spark-TTS...")
    from sparktts.models.audio_tokenizer import BiCodecTokenizer
    audio_tokenizer = BiCodecTokenizer(LOCAL_SNAPSHOT_DIR, "cuda")
else:
    print("audio_tokenizer موجود بالفعل.")

# رجّع موديل فك التوكنات على CUDA لو كان اتحرك للـ CPU
try:
    audio_tokenizer.model.to("cuda")
except Exception as e:
    print("Warning: could not move audio_tokenizer.model to cuda:", repr(e))

def extract_spark_tokens(training_text):
    global_matches = re.findall(r"<\|bicodec_global_(\d+)\|>", training_text)
    semantic_matches = re.findall(r"<\|bicodec_semantic_(\d+)\|>", training_text)
    return global_matches, semantic_matches

def detokenize_training_sample(idx, output_prefix="reconstructed_check"):
    text = dataset[idx]["text"]

    print("=" * 90)
    print("Index:", idx)
    print("Text preview:")
    print(text[:500])

    global_matches, semantic_matches = extract_spark_tokens(text)

    print("\nGlobal tokens:", len(global_matches))
    print("Semantic tokens:", len(semantic_matches))

    if len(global_matches) == 0 or len(semantic_matches) == 0:
        raise ValueError(
            "مفيش Spark audio tokens في dataset[idx]['text'].\n"
            "ده معناه إنك لسه على الداتا الخام أو خلية التوكنيزاشن مرجعتش training_text صح.\n"
            "لا تبدأ التدريب قبل ما يتحل."
        )

    pred_global_ids = torch.tensor(
        [int(x) for x in global_matches],
        dtype=torch.long,
        device="cuda",
    ).unsqueeze(0)

    pred_semantic_ids = torch.tensor(
        [int(x) for x in semantic_matches],
        dtype=torch.long,
        device="cuda",
    ).unsqueeze(0)

    with torch.no_grad():
        wav = audio_tokenizer.detokenize(pred_global_ids, pred_semantic_ids)

    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().numpy()

    wav = np.squeeze(wav)

    sample_rate = audio_tokenizer.config.get("sample_rate", 16000)
    output_path = f"{output_prefix}_{idx}.wav"

    sf.write(output_path, wav, sample_rate)

    print("Saved:", output_path)
    print("Reconstructed duration:", round(len(wav) / sample_rate, 2), "seconds")

    display(IPythonAudio(wav, rate=sample_rate))

    return {
        "idx": idx,
        "global_tokens": len(global_matches),
        "semantic_tokens": len(semantic_matches),
        "duration": len(wav) / sample_rate,
        "output_path": output_path,
    }

# فحص أول عينة
results = []
results.append(detokenize_training_sample(0, "reconstructed_sample"))

# فحص 3 عينات عشوائية
random_indices = random.sample(range(len(dataset)), k=min(3, len(dataset)))

for idx in random_indices:
    results.append(detokenize_training_sample(idx, "reconstructed_sample"))

print("\nSummary:")
for r in results:
    print(
        f"idx={r['idx']} | "
        f"global={r['global_tokens']} | "
        f"semantic={r['semantic_tokens']} | "
        f"duration={r['duration']:.2f}s | "
        f"file={r['output_path']}"
    )

LOCAL_SNAPSHOT_DIR = Spark-TTS-0.5B
audio_tokenizer غير موجود. سيتم تحميله من Spark-TTS...
Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb
Index: 0
Text preview:
<|task_tts|><|start_content|>لِأَنَّهُ لَا يَرَى أَنَّهُ عَلَى السَّفَهِ ثُمَّ مِنْ بَعْدِ ذَلِكَ حَدِيثٌ مُنْتَشِرٌ<|end_content|><|start_global_token|><|bicodec_global_2563|><|bicodec_global_2099|><|bicodec_global_880|><|bicodec_global_3375|><|bicodec_global_2400|><|bicodec_global_2380|><|bicodec_global_1044|><|bicodec_global_1607|><|bicodec_global_3535|><|bicodec_global_3365|><|bicodec_global_3635|><|bicodec_global_3248|><|bicodec_global_815|><|bicodec_global_3605|><|bicodec_global_3315|><|bi

Global tokens: 32
Semantic tokens: 264
Saved: reconstructed_sample_0.wav
Reconstructed duration: 5.28 seconds


Index: 1256
Text preview:
<|task_tts|><|start_content|>وَمَا شَيْءٌ إذَا فَكَّرْتَ فِيهِ بِأَذْهَبَ لِلْمُرُوءَةِ وَالْجَمَالِ<|end_content|><|start_global_token|><|bicodec_global_3243|><|bicodec_global_1073|><|bicodec_global_2824|><|bicodec_global_3391|><|bicodec_global_1577|><|bicodec_global_333|><|bicodec_global_2081|><|bicodec_global_357|><|bicodec_global_3275|><|bicodec_global_2086|><|bicodec_global_1339|><|bicodec_global_4004|><|bicodec_global_569|><|bicodec_global_3585|><|bicodec_global_3315|><|bicodec_global_633|

Global tokens: 32
Semantic tokens: 258
Saved: reconstructed_sample_1256.wav
Reconstructed duration: 5.16 seconds


Index: 8342
Text preview:
<|task_tts|><|start_content|>وَقَالَ سُلَيْمَانُ بْنُ عَبْدِ الْمَلِكِ وَقَالَ سُلَيْمَانُ بْنُ عَبْدِ الْمَلِكِ فِي مَجْلِسِهِ<|end_content|><|start_global_token|><|bicodec_global_2667|><|bicodec_global_1073|><|bicodec_global_2145|><|bicodec_global_3647|><|bicodec_global_2646|><|bicodec_global_2378|><|bicodec_global_2145|><|bicodec_global_72|><|bicodec_global_3571|><|bicodec_global_33|><|bicodec_global_1091|><|bicodec_global_3903|><|bicodec_global_804|><|bicodec_global_3605|><|bicodec_global_32

Global tokens: 32
Semantic tokens: 190
Saved: reconstructed_sample_8342.wav
Reconstructed duration: 3.8 seconds


Index: 3543
Text preview:
<|task_tts|><|start_content|>إنَّ الْحِكْمَةَ تَزِيدُ الشَّرِيفَ شَرَفًا وَتَرْفَعُ الْعَبْدَ الْمَمْلُوكَ حَتَّى تُجْلِسَهُ مَجَالِسَ الْمُلُوكِ<|end_content|><|start_global_token|><|bicodec_global_3695|><|bicodec_global_2099|><|bicodec_global_3019|><|bicodec_global_3627|><|bicodec_global_3140|><|bicodec_global_1101|><|bicodec_global_3442|><|bicodec_global_669|><|bicodec_global_2763|><|bicodec_global_2309|><|bicodec_global_4023|><|bicodec_global_4090|><|bicodec_global_565|><|bicodec_global_3600

Global tokens: 32
Semantic tokens: 342
Saved: reconstructed_sample_3543.wav
Reconstructed duration: 6.84 seconds



Summary:
idx=0 | global=32 | semantic=264 | duration=5.28s | file=reconstructed_sample_0.wav
idx=1256 | global=32 | semantic=258 | duration=5.16s | file=reconstructed_sample_1256.wav
idx=8342 | global=32 | semantic=190 | duration=3.80s | file=reconstructed_sample_8342.wav
idx=3543 | global=32 | semantic=342 | duration=6.84s | file=reconstructed_sample_3543.wav


# **تثبيت إعدادات الدقة أثناء التدريب**

**الخلية دي هدفها تقليل الأخطاء التقنية أثناء التدريب، وجعل التشغيل أكثر ثباتاً.**


In [ ]:
def force_accelerate_no_mixed_precision():
    """Block Accelerate from auto-selecting bf16 on T4 via 4 layered mechanisms."""
    import os
    from pathlib import Path

    # 1) Explicit env vars
    os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
    os.environ["ACCELERATE_USE_CPU"] = "false"
    os.environ["ACCELERATE_CONFIG_FILE"] = "/tmp/accelerate_no_bf16.yaml"

    # 2) Explicit Accelerate config file (kills any silent auto-detection)
    Path("/tmp/accelerate_no_bf16.yaml").write_text(
        'compute_environment: LOCAL_MACHINE\n'
        'distributed_type: "NO"\n'
        'mixed_precision: "no"\n'
        'num_machines: 1\n'
        'num_processes: 1\n'
        'use_cpu: false\n',
        encoding="utf-8",
    )

    # 3) Monkey-patch Accelerator.__init__  — this is the one that actually
    # catches transformers/TRL internally constructing an Accelerator with
    # mixed_precision='bf16' inferred from the model dtype.
    import accelerate
    _orig_init = accelerate.Accelerator.__init__

    def _patched_init(self, *args, **kwargs):
        kwargs["mixed_precision"] = "no"
        return _orig_init(self, *args, **kwargs)

    accelerate.Accelerator.__init__ = _patched_init

    try:
        import transformers.trainer
        transformers.trainer.Accelerator.__init__ = _patched_init
    except Exception as e:
        print("transformers.trainer patch skipped:", repr(e))

    # 4) Clear any AcceleratorState left over from earlier cells/runs
    try:
        from accelerate.state import AcceleratorState
        try:
            AcceleratorState._reset_state(reset_partial_state=True)
        except TypeError:
            AcceleratorState._reset_state()
    except Exception as e:
        print("AcceleratorState reset skipped:", repr(e))

    print("Accelerate forced to mixed_precision=no.")

force_accelerate_no_mixed_precision()

Accelerate forced to mixed_precision=no.


# **إعدادات التدريب وتجهيز المدرب**

**في الخلية دي بنحدد إعدادات التدريب الأساسية، وبعدها ننشئ المدرب اللي هيبدأ عملية الفاين تيونينج.**

In [ ]:
from trl import SFTConfig, SFTTrainer

FINETUNED_MODEL_ID = "testbb/spark-tts-clartts-arabic-v1"

training_args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps  = 4,
    warmup_steps                 = 5,
    num_train_epochs             = 1,
    learning_rate                = 2e-5,

    fp16                         = False,
    bf16                         = False,

    logging_steps                = 100,

    # Important for free Colab. It saves local checkpoints every 250 steps.
    save_steps                   = 1000,
    save_total_limit             = 1,

    push_to_hub                  = True,
    hub_model_id                 = FINETUNED_MODEL_ID,
    hub_strategy                 = "end",

    optim                        = "adamw_8bit",
    weight_decay                 = 0.001,
    lr_scheduler_type            = "linear",
    seed                         = 3407,
    output_dir                   = FINETUNED_MODEL_ID.split("/")[-1],
    report_to                    = "none",

    eval_strategy                = "no",
    packing                      = False,
    max_length                   = max_seq_length,
)

# Extra safety: keep all mixed precision disabled on T4.
training_args.fp16 = False
training_args.bf16 = False

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = training_args,
)

print("Trainer ready. fp16=", training_args.fp16, "bf16=", training_args.bf16)
print("Fine-tuned model will be pushed to:", FINETUNED_MODEL_ID)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/9500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer ready. fp16= False bf16= True
Fine-tuned model will be pushed to: testbb/spark-tts-clartts-arabic-v1


# **بدء التدريب**

**هنا يبدأ الفاين تيونينج فعلياً.**

**الموديل هيبدأ يمر على الداتا، يشوف النص والتوكنز الصوتية المقابلة له، ويحاول يتعلم العلاقة بينهم.**

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint=False)
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,500 | Num Epochs = 1 | Total steps = 1,188
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 506,634,112 of 506,634,112 (100.00% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
100,7.132300
200,5.747800
300,5.562500
400,5.394500
500,5.185700
600,5.050600
700,4.961800
800,4.904400
900,4.873000
1000,4.829700


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
TrainOutput(global_step=1188, training_loss=5.2773481426817, metrics={'train_runtime': 4634.6272, 'train_samples_per_second': 2.05, 'train_steps_per_second': 0.256, 'total_flos': 6252184392799488.0, 'train_loss': 5.2773481426817, 'epoch': 1.0})


# **حفظ ورفع الموديل النهائي**

**بعد انتهاء التدريب، بنحفظ الموديل الناتج.**

**الحفظ المحلي داخل كولاب مفيد مؤقتاً، لكن ملفات كولاب ممكن تختفي بعد انتهاء الجلسة.**

**عشان كده بنرفع الموديل على هاجينج فيس، عشان نقدر نرجع له لاحقاً ونستخدمه في التوليد أو نكمل عليه تجارب أخرى.**

In [ ]:
#@title Save And Push Final Full Fine-Tuned Model

from huggingface_hub import HfApi

FINETUNED_MODEL_ID = "testbb/spark-tts-clartts-arabic-v1" # بدل اسم اليوزر هنا
LOCAL_FINAL_DIR = "spark-tts-clartts-arabic-v1-final"

model.save_pretrained(LOCAL_FINAL_DIR)
tokenizer.save_pretrained(LOCAL_FINAL_DIR)

print("Saved final model locally to:", LOCAL_FINAL_DIR)

model.push_to_hub(
    FINETUNED_MODEL_ID,
    private=True,
    token=HF_TOKEN,
)

tokenizer.push_to_hub(
    FINETUNED_MODEL_ID,
    private=True,
    token=HF_TOKEN,
)

print("Pushed final model to:", FINETUNED_MODEL_ID)

api = HfApi(token=HF_TOKEN)
files = api.list_repo_files(FINETUNED_MODEL_ID, repo_type="model")

print("Files on Hub:")
for f in files:
    print(" -", f)

Saved final model locally to: spark-tts-clartts-arabic-v1-final


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...abic-v1/model.safetensors:   0%|          | 1.09MB / 2.03GB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-arabic-v1/tokenizer.json: 100%|##########| 14.1MB / 14.1MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed final model to: testbb/spark-tts-clartts-arabic-v1
Files on Hub:
 - .gitattributes
 - README.md
 - added_tokens.json
 - chat_template.jinja
 - config.json
 - generation_config.json
 - merges.txt
 - model.safetensors
 - special_tokens_map.json
 - tokenizer.json
 - tokenizer_config.json
 - vocab.json


# **تجربة توليد صوت بعد التدريب**

**في الخلية دي بنختبر المودل بعد الفاين تيونينج.**

In [ ]:
#@title Interactive Arabic Speech Test

import os
import re
import sys
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio as IPythonAudio, display
from huggingface_hub import snapshot_download

# Use the currently loaded model.
# Before fine-tuning, this tests the base Spark-TTS model.
# After fine-tuning, this tests your fine-tuned Arabic model.
model.eval()
torch.cuda.empty_cache()

if "LOCAL_SNAPSHOT_DIR" not in globals():
    if os.path.exists("Spark-TTS-0.5B"):
        LOCAL_SNAPSHOT_DIR = "Spark-TTS-0.5B"
    else:
        LOCAL_SNAPSHOT_DIR = snapshot_download(
            "SparkAudio/Spark-TTS-0.5B",
            local_dir="Spark-TTS-0.5B"
        )

if "Spark-TTS" not in sys.path:
    sys.path.append("Spark-TTS")

if "audio_tokenizer" not in globals():
    from sparktts.models.audio_tokenizer import BiCodecTokenizer
    audio_tokenizer = BiCodecTokenizer(LOCAL_SNAPSHOT_DIR, "cuda")
else:
    try:
        audio_tokenizer.model.to("cuda")
    except Exception as e:
        print("Warning while moving audio_tokenizer to cuda:", repr(e))

def normalize_text_for_tts_arabic(text):
    text = "" if text is None else str(text)
    text = text.replace("ـ", "")
    text = text.replace("\u200f", "")
    text = text.replace("\u200e", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

@torch.inference_mode()
def generate_arabic_speech(
    text,
    temperature=0.20,
    top_k=30,
    top_p=0.80,
    max_new_audio_tokens=900,
    output_path="arabic_interactive_test.wav",
):
    text = normalize_text_for_tts_arabic(text)

    prompt = "".join([
        "<|task_tts|>",
        "<|start_content|>",
        text,
        "<|end_content|>",
        "<|start_global_token|>",
    ])

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print("Input text:")
    print(text)
    print("\nGenerating tokens...")

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_audio_tokens,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        pad_token_id=tokenizer.eos_token_id,
    )

    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
    predicts_text = tokenizer.batch_decode(new_tokens, skip_special_tokens=False)[0]

    global_matches = re.findall(r"<\|bicodec_global_(\d+)\|>", predicts_text)
    semantic_matches = re.findall(r"<\|bicodec_semantic_(\d+)\|>", predicts_text)

    print("Generated global tokens  :", len(global_matches))
    print("Generated semantic tokens:", len(semantic_matches))

    if len(global_matches) == 0 or len(semantic_matches) == 0:
        print("\nRaw generated text preview:")
        print(predicts_text[:1500])
        raise ValueError("The model did not generate enough audio tokens.")

    pred_global_ids = torch.tensor(
        [int(x) for x in global_matches],
        dtype=torch.long,
        device="cuda",
    ).unsqueeze(0)

    pred_semantic_ids = torch.tensor(
        [int(x) for x in semantic_matches],
        dtype=torch.long,
        device="cuda",
    ).unsqueeze(0)

    print("Detokenizing audio...")

    audio_tokenizer.model.to("cuda")

    wav = audio_tokenizer.detokenize(
        pred_global_ids,
        pred_semantic_ids,
    )

    if isinstance(wav, torch.Tensor):
        wav = wav.detach().cpu().numpy()

    wav = np.squeeze(wav)

    sample_rate = audio_tokenizer.config.get("sample_rate", 16000)

    sf.write(output_path, wav, sample_rate)

    print("Saved to:", output_path)
    print("Duration:", round(len(wav) / sample_rate, 2), "seconds")

    display(IPythonAudio(wav, rate=sample_rate))

    return wav

# اكتب الجملة هنا وقت التشغيل.
input_text = input("اكتب الجملة العربية التي تريد تجربتها: ")

generated_waveform = generate_arabic_speech(
    input_text,
    temperature=0.20,
    top_k=30,
    top_p=0.80,
    max_new_audio_tokens=900,
    output_path="arabic_interactive_test.wav",
)

اكتب الجملة العربية التي تريد تجربتها: قناة مدرسة الذكاء الاصطناعي
Input text:
قناة مدرسة الذكاء الاصطناعي

Generating tokens...
Generated global tokens  : 32
Generated semantic tokens: 133
Detokenizing audio...
Saved to: arabic_interactive_test.wav
Duration: 2.66 seconds


**What this cell does:** Prints how long training took and how much GPU memory it used.

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

4242.6971 seconds used for training.
70.71 minutes used for training.
Peak reserved memory = 15.916 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 33.487 %.
Peak reserved memory for training % of max memory = 0.0 %.


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>
